In [15]:
import sys
import os

# Walk up from notebooks/ to the project root (SmartAngelAPI/)
sys.path.insert(0, os.path.abspath(".."))

In [16]:
from client.connection import get_session

session = get_session()
print("auth_token   :", session["auth_token"][:30], "...")
print("refresh_token:", session["refresh_token"][:30], "...")
print("feed_token   :", session["feed_token"][:20], "...")

✓ Using cached session.
auth_token   : eyJhbGciOiJIUzUxMiJ9.eyJ1c2Vyb ...
refresh_token: eyJhbGciOiJIUzUxMiJ9.eyJ0b2tlb ...
feed_token   : eyJhbGciOiJIUzUxMiJ9 ...


In [ ]:
from client.historical import get_candle_data

nifty_15m = get_candle_data(
    auth_token=session["auth_token"],
    symbol_token="99926000",          # from get_symbol_token()
    interval="FIFTEEN_MINUTE",
    from_date="2026-05-25 09:15",
    to_date="2026-05-30 15:30",
    exchange="NSE",
)

nifty_1d = get_candle_data(
    auth_token=session["auth_token"],
    symbol_token="99926000",          # from get_symbol_token()
    interval="ONE_DAY",
    from_date="2026-05-01 09:15",
    to_date="2026-05-30 15:30",
    exchange="NSE",
)



✓ Fetched 100 candles | Token: 99926000 | Interval: FIFTEEN_MINUTE | 2026-05-08 09:15 → 2026-05-13 15:30
                         open      high       low     close  volume
timestamp                                                          
2026-05-08 09:15:00  24233.65  24253.80  24158.15  24244.15       0
2026-05-08 09:30:00  24244.30  24244.95  24167.75  24167.75       0
2026-05-08 09:45:00  24169.05  24199.65  24138.40  24198.45       0
2026-05-08 10:00:00  24199.15  24213.70  24175.10  24211.30       0
2026-05-08 10:15:00  24213.80  24243.95  24181.25  24238.05       0
...                       ...       ...       ...       ...     ...
2026-05-13 14:15:00  23516.50  23522.40  23469.80  23491.35       0
2026-05-13 14:30:00  23491.05  23504.40  23465.20  23467.50       0
2026-05-13 14:45:00  23468.20  23490.85  23446.65  23479.20       0
2026-05-13 15:00:00  23476.75  23478.70  23402.85  23425.70       0
2026-05-13 15:15:00  23425.40  23436.95  23394.35  23428.70       0

[100 rows 

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
# 1. Load your local environment variables
load_dotenv()

# 2. Verify the key is loaded (Optional check)
if "GEMINI_API_KEY" in os.environ:
    print("✅ API Key successfully loaded into environment.")
else:
    print("❌ API Key not found. Check your .env file path.")

# 3. Initialize the Gemini Client
# The client automatically extracts the key from os.environ["GEMINI_API_KEY"]
client = genai.Client()

csv_15m = nifty_15m.to_csv(index=False)
csv_1d = nifty_1d.to_csv(index=False)

# 4. Construct the precise prompt
user_prompt = f"""
Analyze the following multi-timeframe candle data for NIFTY 50. 
Identify the macro trend from the Daily data, and look for immediate momentum or entry signals in the 15-minute data.

### 1-Day Candles (Macro Trend)
{csv_1d}

### 15-Minute Candles (Micro Price Action)
{csv_15m}

Please provide:
1. A brief summary of the multi-timeframe alignment.
2. Key support and resistance levels based on this data.
3. A directional bias for the upcoming session.
"""

print("Sending data to Gemini...\n")

# 5. Execute the API call with specific configurations
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=user_prompt,
    config=types.GenerateContentConfig(
        # System instructions anchor the model's behavior
        system_instruction="You are an expert algorithmic trading assistant. Provide concise, data-driven technical analysis without financial disclaimers.",
        # Low temperature for analytical, deterministic responses
        temperature=0.1, 
    )
)

print(response.text)

✅ API Key successfully loaded into environment.

--- Testing Text Generation ---
Here is a 3-bullet-point summary of why developers love Python:

* **Clean and Readable Syntax:** Python’s simple, English-like syntax allows developers to write less code to achieve more. This high readability makes it easy to learn, write, and maintain, drastically speeding up development time.
* **Vast and Powerful Ecosystem:** It boasts a massive library of pre-built packages and frameworks (like TensorFlow, Django, and Pandas), making it the premier language for cutting-edge fields like artificial intelligence, data science, and web development.
* **Massive, Supportive Community:** With millions of active users worldwide, developers have access to endless tutorials, robust documentation, and quick troubleshooting help, ensuring they never have to solve a problem from scratch.
